In [1]:
from pyspark.sql import SparkSession, functions as F
import sys; sys.executable
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point
from pyspark.sql.functions import col, when
import json

/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.1-CAPI-1.17.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/var/folders/l_/flqtngrd05d8qnbgmbtshrx00000gn/T/ipykernel_27825/892020361.py:4: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Assignment 2")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)

23/09/20 12:03:24 WARN Utils: Your hostname, Cherrys-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1; using 10.13.146.96 instead (on interface en0)
23/09/20 12:03:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/09/20 12:03:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Read in the raw PTV data, the data was ordered from the Victoria State Government and could not be scraped from the internet

In [18]:
gdf_bus_stop = gpd.read_file('../data/landing/PTV/PTV_METRO_BUS_STOP.shp')
gdf_train_station = gpd.read_file('../data/landing/PTV/PTV_METRO_TRAIN_STATION.shp')
gdf_tram_stop = gpd.read_file('../data/landing/PTV/PTV_METRO_TRAM_STOP.shp')
gdf_regional_train_station = gpd.read_file('../data/landing/PTV/PTV_REGIONAL_TRAIN_STATION.shp')

Read in the raw postcode data, the data was ordered from the Victoria State Government and could not be scraped from the internet

In [28]:
gdf_postcode = gpd.read_file('../data/landing/PTV/POSTCODE_POLYGON.shp')

In [20]:
# Drop columns "STOP_ID" and "ROUTEUSSP" from the GeoDataFrame
gdf_train_station = gdf_train_station.drop(columns=["STOP_ID", "ROUTEUSSP"])
gdf_regional_train_station = gdf_regional_train_station.drop(columns=["STOP_ID"])
gdf_bus_stop = gdf_bus_stop.drop(columns=["STOP_ID", "ROUTEUSSP"])
gdf_tram_stop = gdf_tram_stop.drop(columns=["STOP_ID", "ROUTEUSSP"])

In [29]:
# Drop unneeded columns from the postcode data
gdf_postcode = gdf_postcode.drop(columns=["UFI", "PFI", "PFI_CR", "UFI_CR", "TASK_ID", "UFI_OLD"])

In [45]:
# Perform a spatial join to assign postal codes to stations
stations_with_postcodes = gpd.sjoin(gdf_train_station, gdf_postcode, how="left", op="within")
regional_stations_with_postcodes = gpd.sjoin(gdf_regional_train_station, gdf_postcode, how="left", op="within")
bus_stops_with_postcodes = gpd.sjoin(gdf_bus_stop, gdf_postcode, how="left", op="within")

stations_with_postcodes = stations_with_postcodes.drop(columns=["index_right"])
regional_stations_with_postcodes = regional_stations_with_postcodes.drop(columns=["index_right"])
bus_stops_with_postcodes = bus_stops_with_postcodes.drop(columns=["index_right"])

/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3377: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):
/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3377: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):
/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3377: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):


In [40]:
# Define the file paths for saving the GeoDataFrames in the data folder
data_folder = '../data/raw/ptv/'

# Save the GeoDataFrames to files in the data folder
gdf_bus_stop.to_file(data_folder + 'PTV_METRO_BUS_STOP.geojson', driver='GeoJSON')
gdf_train_station.to_file(data_folder + 'PTV_METRO_TRAIN_STATION.geojson', driver='GeoJSON')
gdf_tram_stop.to_file(data_folder + 'PTV_METRO_TRAM_STOP.geojson', driver='GeoJSON')
gdf_regional_train_station.to_file(data_folder + 'PTV_REGIONAL_TRAIN_STATION.geojson', driver='GeoJSON')

gdf_postcode.to_file(data_folder + 'POSTCODE.geojson', driver='GeoJSON')

stations_with_postcodes.to_file(data_folder + 'PTV_TRAIN_STATION_POSTCODE.geojson', driver='GeoJSON')

In [46]:
# Save the GeoDataFrame to a CSV file
stations_with_postcodes.to_csv(data_folder + 'stations_with_postcodes.csv', index=False)
regional_stations_with_postcodes.to_csv(data_folder + 'regional_stations_with_postcodes.csv', index=False)
bus_stops_with_postcodes.to_csv(data_folder + 'bus_stops_with_postcodes.csv', index=False)